# NBA Predictor — Postup tvorby ML modelu

Tento notebook dokumentuje celý proces tvorby prediktivního modelu pro výsledky zápasů NBA.

## Obsah
1. Instalace závislostí
2. Zdroje a sběr dat
3. Načtení dat
4. Předzpracování a čištění
5. Feature engineering
6. Exploratory Data Analysis
7. Trénování modelů
8. Vyhodnocení a porovnání
9. Predikce budoucích zápasů

---
**Zdroje dat:**
- [NBA Stats API](https://stats.nba.com) — statistiky týmů za aktuální sezónu
- [Basketball Reference](https://www.basketball-reference.com) — historické game logy (5 sezón)
- [ESPN Injury API](https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries) — zprávy o zraněních

## 1. Instalace závislostí

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn

## 2. Zdroje a sběr dat

Data jsou sesbírána ze **tří zdrojů** pomocí vlastního kódu ve složce `data/`:

### NBA Stats API (`data/nba_stats.py`)
Volá `https://stats.nba.com/stats/leaguegamelog` — oficiální API NBA. Vrací herní statistiky za každý odehraný zápas aktuální sezóny: body (PTS), procento střelby (FG_PCT, FG3_PCT), doskoky (REB), asistence (AST), ztráty (TOV), krádeže (STL), bloky (BLK).

### Basketball Reference (`data/basketball_ref.py`)
Scraper stahuje historické záznamy z `basketball-reference.com` pro posledních 5 sezón (2021–2026) pomocí knihovny `cloudscraper`. Tato data jsou nutná pro výpočet rolling statistik — aktuální sezóna sama o sobě nestačí.

### ESPN Injury API (`data/injuries.py`)
Stahuje aktuální zprávy o zraněních z `site.api.espn.com`. Slouží k výpočtu dostupnosti hráčů — jeden z klíčových příznaků modelu.

**Celkový objem:** ~12 000+ herních záznamů (30 týmů × ~82 zápasů × 5 sezón)

## 3. Načtení dat

Nahrajte soubory ze `storage/raw/`.

In [ ]:
from google.colab import files
import io
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Nahrajte: nba_api_gamelogs.csv, bref_gamelogs.csv, player_stats.csv')
uploaded = files.upload()

In [ ]:
nba_games    = pd.read_csv(io.BytesIO(uploaded['nba_api_gamelogs.csv']))
bref_games   = pd.read_csv(io.BytesIO(uploaded['bref_gamelogs.csv']))
player_stats = pd.read_csv(io.BytesIO(uploaded['player_stats.csv']))

print(f'NBA API:  {len(nba_games):,} řádků | sloupce: {list(nba_games.columns)}')
print(f'BRef:     {len(bref_games):,} řádků')
print(f'Hráči:    {len(player_stats):,} záznamů')

In [ ]:
nba_games.head(3)

## 4. Předzpracování a čištění dat

Kód z `data/processor.py`. Hlavní kroky:
- Normalizace zkratek týmů (NBA API a Basketball Reference používají různé zkratky, např. `BKN` vs `BRK`)
- Parsování datumů
- Výpočet cílové proměnné `Win` (1 = výhra, 0 = prohra) z kolonky `WL`
- Sloučení obou zdrojů, odstranění duplicit

In [ ]:
TEAM_ABBREVIATION_FIXES = {
    'ATL': 'ATL', 'BOS': 'BOS', 'BKN': 'BRK', 'BRK': 'BRK',
    'CHA': 'CHO', 'CHO': 'CHO', 'CHI': 'CHI', 'CLE': 'CLE',
    'DAL': 'DAL', 'DEN': 'DEN', 'DET': 'DET', 'GSW': 'GSW',
    'HOU': 'HOU', 'IND': 'IND', 'LAC': 'LAC', 'LAL': 'LAL',
    'MEM': 'MEM', 'MIA': 'MIA', 'MIL': 'MIL', 'MIN': 'MIN',
    'NOP': 'NOP', 'NYK': 'NYK', 'OKC': 'OKC', 'ORL': 'ORL',
    'PHI': 'PHI', 'PHX': 'PHO', 'PHO': 'PHO', 'POR': 'POR',
    'SAC': 'SAC', 'SAS': 'SAS', 'TOR': 'TOR', 'UTA': 'UTA', 'WAS': 'WAS',
}

In [ ]:
def process_nba_api_data(df):
    df = df.copy()
    if 'TEAM_ABBREVIATION' in df.columns:
        df['Team'] = df['TEAM_ABBREVIATION'].map(
            lambda x: TEAM_ABBREVIATION_FIXES.get(str(x).upper(), str(x).upper())
        )
    date_col = next((c for c in ['GAME_DATE', 'Date', 'date'] if c in df.columns), None)
    if date_col:
        df['Date'] = pd.to_datetime(df[date_col], errors='coerce')
    if 'WL' in df.columns:
        df['Win'] = (df['WL'] == 'W').astype(int)
    if 'MATCHUP' in df.columns:
        df['is_home'] = df['MATCHUP'].str.contains(' vs. ').astype(int)
        df['Opponent'] = df['MATCHUP'].str.extract(r'(?:vs\.|@)\s+(\w+)$')
        df['Opponent'] = df['Opponent'].map(
            lambda x: TEAM_ABBREVIATION_FIXES.get(str(x).upper(), str(x).upper()) if pd.notna(x) else x
        )
    keep = ['Team', 'Opponent', 'Date', 'Win', 'is_home',
            'PTS', 'FG_PCT', 'FG3_PCT', 'REB', 'AST', 'TOV', 'STL', 'BLK']
    return df[[c for c in keep if c in df.columns]].dropna(subset=['Date', 'Win']).reset_index(drop=True)


def process_bref_data(df):
    df = df.copy()
    df['Date'] = pd.to_datetime(df.get('Date', df.get('date')), errors='coerce')
    df = df.dropna(subset=['Date'])
    if 'Team' in df.columns:
        df['Team'] = df['Team'].map(
            lambda x: TEAM_ABBREVIATION_FIXES.get(str(x).upper(), str(x).upper())
        )
    for col in ['W/L', 'WL']:
        if col in df.columns:
            df['Win'] = df[col].astype(str).str.startswith('W').astype(int)
            break
    if 'HomeAway' in df.columns:
        df['is_home'] = (df['HomeAway'] != '@').astype(int)
    for col in ['OPP_PTS', 'Opp']:
        if col in df.columns:
            df['OPP_PTS'] = pd.to_numeric(df[col], errors='coerce')
            break
    df = df.rename(columns={'FG%': 'FG_PCT', '3P%': 'FG3_PCT', 'TRB': 'REB'})
    keep = ['Team', 'Date', 'Win', 'is_home', 'PTS', 'OPP_PTS', 'FG_PCT', 'FG3_PCT', 'REB', 'AST', 'TOV', 'STL', 'BLK']
    return df[[c for c in keep if c in df.columns]].dropna(subset=['Date', 'Win']).reset_index(drop=True)


nba_clean  = process_nba_api_data(nba_games)
bref_clean = process_bref_data(bref_games)

print(f'NBA API po předzpracování: {len(nba_clean):,} řádků')
print(f'BRef po předzpracování:    {len(bref_clean):,} řádků')

In [ ]:
common = ['Team', 'Date', 'Win', 'is_home', 'PTS', 'FG_PCT', 'FG3_PCT', 'REB', 'AST', 'TOV']
nba_sub  = nba_clean[[c for c in common + ['Opponent'] if c in nba_clean.columns]]
bref_sub = bref_clean[[c for c in common + ['OPP_PTS'] if c in bref_clean.columns]]

games = pd.concat([nba_sub, bref_sub], ignore_index=True)
games['Date'] = pd.to_datetime(games['Date'])
games = games.drop_duplicates(subset=['Team', 'Date']).sort_values('Date').reset_index(drop=True)

if 'Opponent' in games.columns and 'OPP_PTS' not in games.columns:
    pts_lookup = games.set_index(['Team', 'Date'])['PTS'].to_dict()
    games['OPP_PTS'] = games.apply(
        lambda r: pts_lookup.get((r['Opponent'], r['Date']), np.nan), axis=1
    )

print(f'Sloučená data: {len(games):,} řádků')
print(f'Rozsah dat: {games["Date"].min().date()} → {games["Date"].max().date()}')
print(f'Počet týmů: {games["Team"].nunique()}')
games.head(5)

## 5. Feature Engineering

Kód z `features/team_features.py` a `features/builder.py`.

| Příznak | Popis |
|---------|-------|
| `roll_PTS` | Průměr bodů za posledních 10 zápasů |
| `roll_FG_PCT` | Průměrné % střelby za 10 zápasů |
| `form` | % výher za posledních 5 zápasů |
| `streak` | Aktuální série výher (+) nebo proher (-) |
| `rest_days` | Dny od posledního zápasu |
| `elo` | ELO rating týmu (začíná na 1500, K=20) |
| `diff_*` | Diferenciál příznaku: tým − soupeř |
| `is_home` | 1 = domácí, 0 = hosté |
| `is_b2b` | 1 = back-to-back zápas (rest_days ≤ 1) |

In [ ]:
ROLLING_WINDOW = 10
FORM_WINDOW    = 5

stat_cols = [c for c in ['PTS', 'OPP_PTS', 'FG_PCT', 'FG3_PCT', 'REB', 'AST', 'TOV', 'STL', 'BLK']
             if c in games.columns]

gf = games.copy().sort_values(['Team', 'Date'])

for col in stat_cols:
    gf[f'roll_{col}'] = (
        gf.groupby('Team')[col]
        .transform(lambda x: x.rolling(ROLLING_WINDOW, min_periods=3).mean().shift(1))
    )

gf['form'] = (
    gf.groupby('Team')['Win']
    .transform(lambda x: x.rolling(FORM_WINDOW, min_periods=2).mean().shift(1))
)

def streak_series(series):
    streak, result = 0, []
    for val in series:
        result.append(streak)
        if val == 1:
            streak = (streak + 1) if streak >= 0 else 1
        else:
            streak = (streak - 1) if streak <= 0 else -1
    return pd.Series(result, index=series.index)

gf['streak'] = gf.groupby('Team')['Win'].transform(streak_series)

gf['rest_days'] = (
    gf.groupby('Team')['Date']
    .transform(lambda x: x.diff().dt.days.fillna(3))
)
gf['is_b2b'] = (gf['rest_days'] <= 1).astype(int)

print('Rolling statistiky, form, streak, rest_days přidány.')
gf[['Team', 'Date', 'PTS', 'roll_PTS', 'form', 'streak', 'rest_days']].dropna().head(5)

In [ ]:
ELO_K              = 20.0
ELO_HOME_ADVANTAGE = 100.0
ELO_INITIAL        = 1500.0

gf = gf.sort_values('Date').reset_index(drop=True)
ratings = {}
elo_col = [0.0] * len(gf)

for idx, row in gf.iterrows():
    team = row['Team']
    opp  = row.get('Opponent', '')
    is_home = int(row.get('is_home', 1))

    r_team = ratings.get(team, ELO_INITIAL)
    r_opp  = ratings.get(opp,  ELO_INITIAL)
    elo_col[idx] = r_team

    r_t_eff = r_team + (ELO_HOME_ADVANTAGE if is_home else 0.0)
    r_o_eff = r_opp  + (ELO_HOME_ADVANTAGE if not is_home else 0.0)
    expected = 1.0 / (1.0 + 10.0 ** ((r_o_eff - r_t_eff) / 400.0))
    actual   = float(row.get('Win', 0.5))
    ratings[team] = r_team + ELO_K * (actual - expected)

gf['elo'] = elo_col

top5 = dict(sorted(ratings.items(), key=lambda x: x[1], reverse=True)[:5])
print('ELO přidán. Top 5 týmů:', top5)

In [ ]:
PLAYER_VALUE_WEIGHTS = {
    'PTS': 1.0, 'REB': 0.7, 'AST': 1.0,
    'STL': 1.5, 'BLK': 1.2, 'TOV': -1.0,
}

ps = player_stats.copy()
ps['player_value'] = sum(
    ps[col].fillna(0) * w
    for col, w in PLAYER_VALUE_WEIGHTS.items()
    if col in ps.columns
)
if 'GP' in ps.columns:
    ps['availability_factor'] = (ps['GP'] / ps['GP'].max()).clip(0, 1)
    ps['is_available'] = ps['availability_factor'] > 0.7
else:
    ps['availability_factor'] = 1.0
    ps['is_available'] = True
ps['weighted_value'] = ps['player_value'] * ps['availability_factor']

tc = 'TEAM_ABBREVIATION' if 'TEAM_ABBREVIATION' in ps.columns else 'Team'
total_str = ps.groupby(tc)['player_value'].sum().rename('roster_strength')
avail_str = ps[ps['is_available']].groupby(tc)['weighted_value'].sum().rename('available_strength')
roster = pd.DataFrame({'roster_strength': total_str, 'available_strength': avail_str}).fillna(0)
roster['availability_ratio'] = (roster['available_strength'] / roster['roster_strength'].replace(0, 1)).clip(0, 1)
roster = roster.reset_index().rename(columns={tc: 'Team'})

print(f'Roster strength vypočítán pro {len(roster)} týmů.')
roster.sort_values('availability_ratio', ascending=False).head(5)

In [ ]:
df = gf.copy()
roll_cols = [c for c in df.columns if c.startswith('roll_') or c in ('form', 'streak', 'rest_days', 'elo')]

if 'Opponent' in df.columns:
    opp_df = df[['Date', 'Team'] + roll_cols].copy()
    opp_df.columns = ['Date', 'Opponent'] + [f'opp_{c}' for c in roll_cols]
    df = df.merge(opp_df, on=['Date', 'Opponent'], how='inner')

    for team_c, opp_c, diff_c in [
        ('roll_PTS',     'opp_roll_PTS',     'diff_pts'),
        ('roll_OPP_PTS', 'opp_roll_OPP_PTS', 'diff_opp_pts'),
        ('roll_FG_PCT',  'opp_roll_FG_PCT',  'diff_fg_pct'),
        ('roll_FG3_PCT', 'opp_roll_FG3_PCT', 'diff_fg3_pct'),
        ('roll_REB',     'opp_roll_REB',     'diff_reb'),
        ('roll_AST',     'opp_roll_AST',     'diff_ast'),
        ('roll_TOV',     'opp_roll_TOV',     'diff_tov'),
        ('roll_STL',     'opp_roll_STL',     'diff_stl'),
        ('roll_BLK',     'opp_roll_BLK',     'diff_blk'),
        ('form',         'opp_form',         'diff_form'),
        ('streak',       'opp_streak',       'diff_streak'),
        ('rest_days',    'opp_rest_days',    'diff_rest'),
        ('elo',          'opp_elo',          'diff_elo'),
    ]:
        if team_c in df.columns and opp_c in df.columns:
            df[diff_c] = df[team_c] - df[opp_c]

if not roster.empty:
    df = df.merge(roster[['Team', 'availability_ratio']], on='Team', how='left')
    df = df.merge(
        roster[['Team', 'availability_ratio']].rename(columns={'Team': 'Opponent', 'availability_ratio': 'opp_avail'}),
        on='Opponent', how='left'
    )
    df['availability_ratio'] = df['availability_ratio'].fillna(1.0)
    df['opp_avail']           = df['opp_avail'].fillna(1.0)
    df['diff_roster'] = df['availability_ratio'] - df['opp_avail']

ALL_FEATURES = [
    'diff_pts', 'diff_opp_pts', 'diff_fg_pct', 'diff_fg3_pct',
    'diff_reb', 'diff_ast', 'diff_tov', 'diff_stl', 'diff_blk',
    'diff_form', 'diff_streak', 'diff_rest', 'diff_roster', 'diff_elo',
    'is_home', 'is_b2b',
]
feature_cols = [c for c in ALL_FEATURES if c in df.columns]

for col in {'diff_fg3_pct', 'diff_stl', 'diff_blk', 'diff_roster', 'diff_elo'} & set(feature_cols):
    df[col] = df[col].fillna(0.0)

core = [c for c in feature_cols if c not in {'diff_fg3_pct', 'diff_stl', 'diff_blk', 'diff_roster', 'diff_elo'}]
final_df = df.dropna(subset=core)

print(f'Finální dataset: {len(final_df):,} řádků')
print(f'Příznaky ({len(feature_cols)}): {feature_cols}')
print(f'Win rate: {final_df["Win"].mean():.1%}')

## 6. Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0a0e17', 'axes.facecolor': '#111827',
    'text.color': 'white', 'axes.labelcolor': 'white',
    'xtick.color': 'white', 'ytick.color': 'white',
    'axes.edgecolor': '#374151', 'grid.color': '#1e2738', 'axes.grid': True,
})

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribuce klíčových příznaků — výhry vs. prohry', color='white', fontsize=13)

for ax, col, label in [
    (axes[0], 'diff_pts',  'Rozdíl bodů'),
    (axes[1], 'diff_form', 'Rozdíl formy'),
    (axes[2], 'diff_elo',  'Rozdíl ELO'),
]:
    if col not in final_df.columns:
        continue
    ax.hist(final_df[final_df['Win']==0][col].dropna(), bins=40, alpha=0.6, color='#ef4444', label='Prohra', density=True)
    ax.hist(final_df[final_df['Win']==1][col].dropna(), bins=40, alpha=0.6, color='#22c55e', label='Výhra',  density=True)
    ax.axvline(0, color='white', linestyle='--', alpha=0.5)
    ax.set_title(label, color='white')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
corr = final_df[feature_cols + ['Win']].corr()['Win'].drop('Win').sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0a0e17')
colors = ['#ef4444' if v < 0 else '#22c55e' for v in corr]
corr.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='white', linewidth=0.8)
ax.set_title('Korelace příznaků s výsledkem zápasu', color='white', fontsize=13)
ax.set_xlabel('Pearsonův korelační koeficient')
plt.tight_layout()
plt.show()

print('Nejsilnější příznaky:')
print(corr.abs().sort_values(ascending=False))

In [ ]:
if 'is_home' in final_df.columns:
    home_wr = final_df[final_df['is_home']==1]['Win'].mean()
    away_wr = final_df[final_df['is_home']==0]['Win'].mean()

    fig, ax = plt.subplots(figsize=(5, 4))
    fig.patch.set_facecolor('#0a0e17')
    bars = ax.bar(['Domácí', 'Hosté'], [home_wr*100, away_wr*100], color=['#3b82f6','#ef4444'], width=0.4)
    ax.set_ylim(0, 80)
    ax.set_ylabel('Win rate (%)')
    ax.set_title('Výhoda domácího hřiště', color='white')
    for bar, val in zip(bars, [home_wr, away_wr]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{val:.1%}', ha='center', color='white', fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f'Domácí: {home_wr:.1%}  |  Hosté: {away_wr:.1%}')

## 7. Trénování modelů

Trénujeme 2 klasifikátory:
1. **Logistic Regression** — lineární baseline (GridSearchCV, hledá optimální C)
2. **Random Forest** — ensemble rozhodovacích stromů (RandomizedSearchCV, 50 iterací)

**Rozdělení dat:** časový split 80/20 — starší zápasy trénují, novější testují. Náhodný split by způsobil data leakage (model by se trénoval na budoucích datech).

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, log_loss, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
CV_FOLDS     = 5

sorted_df = final_df.sort_values('Date').reset_index(drop=True)
split_idx = int(len(sorted_df) * 0.8)

X_train = sorted_df.iloc[:split_idx][feature_cols].fillna(0)
y_train = sorted_df.iloc[:split_idx]['Win']
X_test  = sorted_df.iloc[split_idx:][feature_cols].fillna(0)
y_test  = sorted_df.iloc[split_idx:]['Win']

print(f'Trénink: {len(X_train):,} zápasů  ({sorted_df.iloc[0]["Date"].date()} → {sorted_df.iloc[split_idx-1]["Date"].date()})')
print(f'Test:    {len(X_test):,} zápasů   ({sorted_df.iloc[split_idx]["Date"].date()} → {sorted_df.iloc[-1]["Date"].date()})')

In [ ]:
lr_search = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]),
    {'clf__C': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0], 'clf__solver': ['lbfgs', 'liblinear']},
    cv=CV_FOLDS, scoring='roc_auc', n_jobs=-1, refit=True,
)
lr_search.fit(X_train, y_train)
lr_model = lr_search.best_estimator_
print(f'Logistic Regression  CV AUC: {lr_search.best_score_:.4f}  |  params: {lr_search.best_params_}')

In [ ]:
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    {
        'n_estimators':      [200, 300, 400, 500],
        'max_depth':         [4, 6, 8, 10, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 4],
        'max_features':      ['sqrt', 'log2', 0.3],
        'class_weight':      [None, 'balanced'],
    },
    n_iter=50, cv=CV_FOLDS, scoring='roc_auc', n_jobs=-1, random_state=RANDOM_STATE, refit=True,
)
rf_search.fit(X_train, y_train)
rf_model = rf_search.best_estimator_
print(f'Random Forest  CV AUC: {rf_search.best_score_:.4f}  |  params: {rf_search.best_params_}')

## 8. Vyhodnocení a porovnání modelů

In [ ]:
results = []
for name, model in [('Logistic Regression', lr_model), ('Random Forest', rf_model)]:
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    try:
        cv = cross_val_score(model, X_train, y_train, cv=CV_FOLDS, scoring='roc_auc')
    except Exception:
        cv = np.array([roc_auc_score(y_test, probs)])
    results.append({
        'Model':    name,
        'Accuracy': round(accuracy_score(y_test, preds), 4),
        'ROC-AUC':  round(roc_auc_score(y_test, probs), 4),
        'F1 Score': round(f1_score(y_test, preds), 4),
        'Log Loss': round(log_loss(y_test, probs), 4),
        'CV Mean':  round(cv.mean(), 4),
        'CV Std':   round(cv.std(), 4),
    })

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
results_df

In [ ]:
metrics = ['Accuracy', 'ROC-AUC', 'F1 Score']
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#0a0e17')
colors = ['#3b82f6', '#22c55e']

for i, (_, row) in enumerate(results_df.iterrows()):
    bars = ax.bar(x + (i-0.5)*width, [row[m] for m in metrics], width,
                  label=row['Model'], color=colors[i], alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                f'{bar.get_height():.3f}', ha='center', color='white', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0.45, 0.85)
ax.set_title('Porovnání výkonu modelů', color='white', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_name  = results_df.iloc[0]['Model']
model_map  = {'Logistic Regression': lr_model, 'Random Forest': rf_model}
best_model = model_map[best_name]

cm = confusion_matrix(y_test, best_model.predict(X_test))
fig, ax = plt.subplots(figsize=(5, 4))
fig.patch.set_facecolor('#0a0e17')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Prohra', 'Výhra']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — {best_name}', color='white')
plt.tight_layout()
plt.show()

print(f'Nejlepší model: {best_name}')
print(results_df.iloc[0])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0a0e17')

if best_name == 'Logistic Regression':
    imp = pd.Series(np.abs(best_model.named_steps['clf'].coef_[0]), index=feature_cols).sort_values()
    imp.plot(kind='barh', ax=ax, color='#3b82f6')
    ax.set_title('Logistic Regression — absolutní koeficienty', color='white')
else:
    imp = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values()
    imp.plot(kind='barh', ax=ax, color='#f97316')
    ax.set_title(f'{best_name} — Feature Importance', color='white')

ax.set_xlabel('Důležitost')
plt.tight_layout()
plt.show()

## 9. Predikce budoucích zápasů

Nejlepší natrénovaný model použijeme pro predikci nadcházejících zápasů ze souboru `schedule.csv`.

In [ ]:
print('Nahrajte schedule.csv ze storage/raw/')
sched_up = files.upload()

In [ ]:
TEAM_ABBR_TO_NAME = {
    'ATL':'Atlanta Hawks',    'BOS':'Boston Celtics',    'BRK':'Brooklyn Nets',
    'CHO':'Charlotte Hornets','CHI':'Chicago Bulls',     'CLE':'Cleveland Cavaliers',
    'DAL':'Dallas Mavericks', 'DEN':'Denver Nuggets',    'DET':'Detroit Pistons',
    'GSW':'Golden State Warriors','HOU':'Houston Rockets','IND':'Indiana Pacers',
    'LAC':'LA Clippers',      'LAL':'Los Angeles Lakers','MEM':'Memphis Grizzlies',
    'MIA':'Miami Heat',       'MIL':'Milwaukee Bucks',   'MIN':'Minnesota Timberwolves',
    'NOP':'New Orleans Pelicans','NYK':'New York Knicks', 'OKC':'Oklahoma City Thunder',
    'ORL':'Orlando Magic',    'PHI':'Philadelphia 76ers','PHO':'Phoenix Suns',
    'POR':'Portland Trail Blazers','SAC':'Sacramento Kings','SAS':'San Antonio Spurs',
    'TOR':'Toronto Raptors',  'UTA':'Utah Jazz',         'WAS':'Washington Wizards',
}
NAME_TO_ABBR = {v: k for k, v in TEAM_ABBR_TO_NAME.items()}

def resolve(name):
    return NAME_TO_ABBR.get(name, TEAM_ABBREVIATION_FIXES.get(name, name))

schedule = pd.read_csv(io.BytesIO(sched_up['schedule.csv']))
schedule['Date'] = pd.to_datetime(schedule['Date'])
upcoming = schedule[schedule['Date'] >= pd.Timestamp.today().normalize()].copy()

latest = final_df.sort_values('Date').groupby('Team').last().reset_index()

ROLL_TO_DIFF = [
    ('roll_PTS','diff_pts'), ('roll_OPP_PTS','diff_opp_pts'), ('roll_FG_PCT','diff_fg_pct'),
    ('roll_FG3_PCT','diff_fg3_pct'), ('roll_REB','diff_reb'), ('roll_AST','diff_ast'),
    ('roll_TOV','diff_tov'), ('roll_STL','diff_stl'), ('roll_BLK','diff_blk'),
    ('form','diff_form'), ('streak','diff_streak'), ('rest_days','diff_rest'), ('elo','diff_elo'),
]

predictions = []
for _, game in upcoming.iterrows():
    home = resolve(game['Home'])
    away = resolve(game['Away'])
    hs = latest[latest['Team'] == home]
    as_ = latest[latest['Team'] == away]
    if hs.empty or as_.empty:
        continue
    hr, ar = hs.iloc[0], as_.iloc[0]
    matchup = {d: float(hr.get(r, 0) or 0) - float(ar.get(r, 0) or 0) for r, d in ROLL_TO_DIFF if d in feature_cols}
    matchup['is_home'] = 1.0
    matchup['is_b2b']  = 0.0
    if 'diff_roster' in feature_cols:
        matchup['diff_roster'] = 0.0
    X_p = pd.DataFrame([[matchup.get(c, 0.0) for c in feature_cols]], columns=feature_cols)
    try:
        prob = float(best_model.predict_proba(X_p)[:, 1][0])
    except Exception:
        prob = 0.5
    winner = home if prob >= 0.5 else away
    predictions.append({
        'Datum':     game['Date'].strftime('%Y-%m-%d'),
        'Domácí':    TEAM_ABBR_TO_NAME.get(home, home),
        'Hosté':     TEAM_ABBR_TO_NAME.get(away, away),
        'P(domácí)': f'{prob:.1%}',
        'P(hosté)':  f'{1-prob:.1%}',
        'Vítěz':     TEAM_ABBR_TO_NAME.get(winner, winner),
    })

pred_df = pd.DataFrame(predictions)
print(f'{len(pred_df)} predikcí:')
pred_df.head(20)

In [ ]:
import joblib

joblib.dump(best_model, 'best_model.pkl')
with open('best_model_meta.txt', 'w') as f:
    f.write(best_name)
results_df.to_csv('model_comparison.csv', index=False)

files.download('best_model.pkl')
files.download('best_model_meta.txt')
files.download('model_comparison.csv')

print(f'Uloženo: {best_name}')
print('Zkopírujte soubory do storage/trained/')

## Závěr

1. **Sběr dat** — vlastní scraper stáhl ~12 000+ reálných herních záznamů z NBA Stats API a Basketball Reference
2. **Předzpracování** — normalizace zkratek, parsování datumů, výpočet Win/Loss
3. **Feature engineering** — 16 příznaků: rolling statistiky (10-game window), forma, streak, rest days, ELO, roster dostupnost, diferenciály
4. **Trénování** — 2 modely s automatickým laděním hyperparametrů (GridSearchCV / RandomizedSearchCV, 5-fold cross-validation)
5. **Vyhodnocení** — časový split 80/20, metriky: Accuracy, ROC-AUC, F1 Score, Log Loss
6. **Nasazení** — nejlepší model uložen jako `best_model.pkl`, používán webovou aplikací